# Artifacta + OpenAI Agents SDK — starter

An "agent that produces files" demo: the agent writes a short report, then stores it in Artifacta — no HTTP plumbing, just the MCP tool catalog.

## Run instructions (one line)

Install the extra and export both keys, then run the cells top-to-bottom:

```bash
pip install 'artifacta-mcp[openai-agents]' && export ARTIFACTA_API_KEY=ak_live_... OPENAI_API_KEY=sk-...
```

The Artifacta MCP server is launched **in-process** as a stdio subprocess by `artifacta_mcp_server(...)` — you do **not** need a separate `artifacta-mcp` terminal. (If you prefer to launch it yourself, run `pipx run artifacta-mcp` in another terminal and pass `command="pipx", args=["run", "artifacta-mcp"]`.)

Requires Python 3.10+ and Jupyter with an async-capable kernel (ipykernel ≥ 6 supports top-level `await`).

## 1. Preflight — confirm both keys are set

In [ ]:
import os

assert os.environ.get("ARTIFACTA_API_KEY", "").startswith("ak_live_"), (
    "Set ARTIFACTA_API_KEY (ak_live_...). Get one at https://app.artifacta.io/dashboard/keys"
)
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY for the OpenAI Agents SDK."
print("Keys present \u2713")

## 2. Register Artifacta and run the agent

`artifacta_mcp_server(...)` builds an `agents.mcp.MCPServerStdio` pointed at the published `artifacta-mcp` server. We allow the current working directory so the agent can read the file it writes (`store_artifact`'s `path` is path-confined to this allow-list).

In [ ]:
from agents import Agent, Runner

from artifacta_mcp.openai_agents import artifacta_mcp_server

SESSION_ID = "openai_agents_demo"

async def run_demo():
    # The MCP server runs as a stdio subprocess for the life of this block.
    async with artifacta_mcp_server(allow_path=os.getcwd()) as artifacta:
        agent = Agent(
            name="report-writer",
            instructions=(
                "You produce files and store them in Artifacta using the "
                "store_artifact tool. Always store outputs under the session_id "
                "you are given, and report the resulting artifact id."
            ),
            mcp_servers=[artifacta],
        )
        prompt = (
            "Write a short 5-section report on why a dedicated artifact store "
            "helps AI agents. Save it to a local file named report.md, then "
            f"store that file in Artifacta under session_id='{SESSION_ID}' with "
            "metadata kind=report. Finally, list the artifacts in that session "
            "and tell me the artifact id."
        )
        return await Runner.run(agent, prompt, max_turns=12)

result = await run_demo()
print(result.final_output)

## 3. Assert an artifact was actually created

Don't trust the agent's narration — verify against the API directly via the `artifacta` SDK (the same single HTTP client the MCP server uses).

In [ ]:
from artifacta import Client

client = Client(api_key=os.environ["ARTIFACTA_API_KEY"])
listing = client.list(session_id=SESSION_ID)

assert len(listing) >= 1, (
    f"Expected at least one artifact in session {SESSION_ID!r}; the agent did "
    "not store anything. Check the agent transcript above."
)
print(f"\u2705 {len(listing)} artifact(s) stored in session {SESSION_ID!r}:")
for art in listing:
    print("  -", art)

## Next steps

- Attach Artifacta to an agent you already built with `register(agent, allow_path=...)` instead of constructing the server inline.
- Add `--allow-destructive` (via `allow_destructive=True`) to let the agent mint public share links, delete artifacts, or seal sessions — only with a human in the loop.
- See the [Artifacta MCP docs](https://docs.artifacta.io/mcp) for the full tool surface and the [README](../../README.md) for the LangChain and CrewAI integrations.